In [ ]:
# === Colab / Binder bootstrap — make sure qm_toolkit.py is importable ===
# On a full clone or Binder this is a no-op. On Colab (single-file open) it grabs
# the one helper file every demo imports, so the cells below "just run".
import os, urllib.request
_here = [os.getcwd()] + [os.path.abspath(os.path.join(os.getcwd(), *[os.pardir]*k)) for k in range(1, 6)]
if not any(os.path.isfile(os.path.join(p, "qm_toolkit.py")) for p in _here):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/HogRed/quantum_honestly_public/main/qm_toolkit.py",
        "qm_toolkit.py")
    print("fetched qm_toolkit.py for this session")

# The Two-Slit Mystery

### *Quantum, Honestly* — Episode 1

Hook them with the strangest fact in physics using zero formalism, then plant the IOU 'amplitudes add, then square.' Run All once before presenting so the toolkit is loaded and the demo cells are warm.

In [ ]:
# === SETUP -- run this first (Cell > Run All) ===
%matplotlib inline
import sys, os
# robust: find the folder that has qm_toolkit.py, wherever Jupyter started the kernel
_root = next((d for d in [os.getcwd(), *[os.path.abspath(os.path.join(os.getcwd(), *[os.pardir]*k))
                                         for k in range(1, 6)]]
              if os.path.isfile(os.path.join(d, 'qm_toolkit.py'))), r'C:\Users\jtfai\qm')
sys.path.insert(0, _root)
import numpy as np
import matplotlib.pyplot as plt
import qm_toolkit as qm

rng = np.random.default_rng(1)   # one stream; the E1 build-up cell draws from it
screen = np.linspace(-300, 300, 1201)
fringe = qm.two_slit_intensity(screen, both=True)      # the real (interference) pattern
print("setup ready - qm_toolkit loaded")

## Open a second door… and they vanish

<video src="media/videos/E1_manim/1080p60/DoorParadox.mp4" width="860" controls autoplay loop muted></video>

Left slit only → **100**.  Right slit only → **100**.  Both open → **0**.

Watch one spot. Left only: 100. Right only: 100. So both should be 200, obviously. It's ZERO. Opening a second door made them vanish — impossible for anything classical, and exactly what nature does.

## The natural (wrong) picture

An electron is a **tiny ball**. So it goes through the **left slit OR the right** — and opening a second slit can only **add** hits, never remove them.

> Hold onto that picture. The screen is about to break it.

## First, two things we *do* understand

**Tennis balls** through two gaps → each goes through one gap → **two humps** that just add. No fringes.

**Water waves** through two gaps → crest+crest = bright, crest+trough = **cancel** → **interference fringes**.

*Adding two things can give you less — when they're waves.*

> **Leak:** the electron is *not* a wave of stuff — it borrows the wave's math, not its substance, and always lands as a single dot.

In [ ]:
# diagram-only slide (input hidden in the deck)
qm.interference_schematic();

In [ ]:
# LIVE: if electrons were balls (two humps) vs what really happens (fringes)
def gauss(x, mu, s): return np.exp(-((x - mu)**2) / (2 * s**2))
balls = gauss(screen, -70, 45) + gauss(screen, 70, 45); balls /= balls.max()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
a1.plot(screen, balls, color="#cb4b16"); a1.fill_between(screen, balls, color="#cb4b16", alpha=.15)
a1.set_title("If electrons were balls: two humps")
a2.plot(screen, fringe, color="#2b8cbe"); a2.fill_between(screen, fringe, color="#2b8cbe", alpha=.15)
a2.set_title("What actually happens: fringes")
for ax in (a1, a2): ax.set_xlabel("screen position"); ax.set_yticks([])
plt.show()

In [ ]:
# diagram-only slide (input hidden in the deck)
qm.two_slit_schematic();

## Now: electrons, one at a time

<img src="figures/double_slit_buildup.gif" width="660">

Each arrives as a single **dot** (particle-like)… yet they pile into the **wave's** fringes.

> The bell-shaped fall-off is borrowed single-slit optics (paid off in E7); today we build the interference *logic*.

In [ ]:
# LIVE: drop electrons one at a time -> they pile into the fringes
rng = np.random.default_rng(1)   # reset so re-running this cell replays identically
N = 30000
xs = qm.sample_hits(screen, fringe, shots=N, rng=rng)   # land each with prob ~ |psi|^2
ys = rng.uniform(0, 1, N)                                 # random vertical position
plt.figure(figsize=(9, 4))
plt.scatter(xs, ys, s=2, alpha=0.4, c="#2b8cbe", linewidths=0)
plt.title(f"{N:,} electrons, one at a time"); plt.yticks([]); plt.xlabel("screen position")
plt.show()
# try N = 200 or 2000 to watch the pattern emerge from noise

## The crime scene

| spot | left only | right only | 'particle' guess | the screen |
|---|---|---|---|---|
| dark | 100 | 100 | **200** | **0** |
| bright | 100 | 100 | **200** | **400** |

Brighter **and** darker than the sum. No pile of particles can do that.

## Two more twists

1. Send them **one at a time**, minutes apart — *still* fringes. Each electron interferes with **itself**.
2. Put a detector to see **which slit** — the fringes **vanish**. *Looking* changes the answer.

*(That second one is measurement — Episode 3.)*

In [ ]:
# diagram-only slide (input hidden in the deck)
qm.which_path_schematic();

## The rule underneath: amplitudes, not probabilities

<video src="media/videos/E1_manim/1080p60/AmplitudesNotProbabilities.mp4" width="860" controls autoplay loop muted></video>

Add the **amplitudes** first, *then* square. The cross-term is the interference.

In [ ]:
# diagram-only slide (input hidden in the deck)
qm.amplitude_addition_schematic();

In [ ]:
# LIVE: why SQUARE? add amplitudes then square -- the only rule that matches the screen
psi1, psi2 = 1 + 0j, -1 + 0j                 # two amplitudes meeting at a DARK fringe
print("add amplitudes, THEN square :", abs(psi1 + psi2)**2, " -> dark  (matches the screen)")
print("square THEN add             :", abs(psi1)**2 + abs(psi2)**2, " -> bright (WRONG)")

nofr = qm.two_slit_intensity(screen, both=False)         # add probabilities -> no fringes
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
a1.plot(screen, nofr, color="#cb4b16");  a1.set_title("add probabilities: no fringes")
a2.plot(screen, fringe, color="#2b8cbe"); a2.set_title("add amplitudes, then square: fringes")
for ax in (a1, a2): ax.set_xlabel("screen position"); ax.set_yticks([])
plt.show()

## Particle or wave?

<video src="media/videos/E1_manim/1080p60/ParticleOrWaveNeither.mp4" width="860" controls autoplay loop muted></video>

**Neither.** A new kind of thing — a **wavefunction**.

## Recap → next time

- Indivisible dots, wave pattern.
- A second slit can make a spot **darker**.
- Each electron interferes with **itself**; **looking** erases it.
- One rule: **amplitudes add, then square.**

**Next — E2: The Quantum Coin.** We build the machine that does this, out of *two numbers.*

Close on the reframe and the hook to E2. The whole series pays back the 'amplitudes add, then square' IOU.